# 03 - Resource Recommender
**Task:** for a skill in the user's roadmap, pick the best 2-3 resources for *this* user.

**Approach (content-based filtering + feedback re-ranking):** each resource gets a score from
- **level fit**: does it match the user's level?
- **time fit**: how many days it takes at the user's daily study time
- **type fit**: videos/courses suit beginners, docs/practice suit intermediate users
- **feedback**: share of learners who marked it helpful (smoothed, so 1 vote does not dominate)
- **free**: free resources get a small bonus

Why not a trained model? There is no real user feedback yet (cold start). A transparent score works from day 1,
every recommendation can be explained, and real feedback from the website improves it automatically.

In [1]:
import pandas as pd
from recommender import ResourceRecommender, DEFAULT_WEIGHTS, LEVELS

skills = pd.read_csv("../data/skills.csv")
resources = pd.read_csv("../data/resources.csv")
names = dict(zip(skills.skill_id, skills.skill_name))
rec = ResourceRecommender(resources)
print(len(resources), "resources for", resources.skill_id.nunique(), "skills")
resources.groupby("skill_id").size().value_counts().rename("skills with N resources")

84 resources for 42 skills


2    42
Name: skills with N resources, dtype: int64

**Note:** each skill has only 2 resources right now, so "ranking" has little room to show its value.
Adding 4-5 resources per skill (different levels and types) is the best way to improve this component.

In [2]:
resources["level"].value_counts()

level
beginner        55
intermediate    27
advanced         2
Name: count, dtype: int64

## 1. See it work: same skill, different learners

In [3]:
def show(skill_id, level, minutes):
    print(f"\n{names[skill_id]} | level={level}, {minutes} min/day")
    for i, r in enumerate(rec.recommend(skill_id, level, minutes), 1):
        print(f"  {i}. {r['title']} [{r['type']}, {r['level']}] score={r['score']}")
        print("     -", "; ".join(r["reasons"]))

show("S31", "beginner", 60)
show("S31", "intermediate", 240)
show("S24", "beginner", 120)
show("S24", "intermediate", 120)


Algorithms | level=beginner, 60 min/day
  1. CS50 (Harvard) [course, beginner] score=0.752
     - matches your level; about 20.0 days at 60 min/day
  2. GeeksforGeeks: Algorithms [docs, intermediate] score=0.482
     - a slight stretch above your level; about 10.0 days at 60 min/day

Algorithms | level=intermediate, 240 min/day
  1. GeeksforGeeks: Algorithms [docs, intermediate] score=0.847
     - matches your level; about 2.5 days at 240 min/day
  2. CS50 (Harvard) [course, beginner] score=0.627
     - a bit easier than your level; about 5.0 days at 240 min/day

Machine Learning Basics | level=beginner, 120 min/day
  1. Kaggle Learn: Intro to Machine Learning [course, beginner] score=0.865
     - matches your level; about 1.5 days at 120 min/day
  2. scikit-learn User Guide [docs, intermediate] score=0.517
     - a slight stretch above your level; about 5.0 days at 120 min/day

Machine Learning Basics | level=intermediate, 120 min/day
  1. scikit-learn User Guide [docs, intermediate]

## 2. Sanity tests
Simple rules a sensible recommender must follow. Each line should print PASS.

In [4]:
def check(name, cond):
    print("PASS" if cond else "FAIL", "-", name)

# 1. unknown skill -> nothing
check("unknown skill returns an empty list", rec.recommend("S99") == [])

# 2. top_k respected, scores between 0 and 1
allr = [r for s in skills.skill_id for r in rec.recommend(s, "beginner", 120, top_k=1)]
check("all scores are between 0 and 1", all(0 <= r["score"] <= 1 for r in allr))
check("top_k=1 returns one resource", all(len(rec.recommend(s, "beginner", 120, top_k=1)) == 1 for s in skills.skill_id))

# 3. level matters: a beginner is shown a beginner resource first whenever the skill has one
bad = []
for s in skills.skill_id:
    rows = resources[resources.skill_id == s]
    if (rows.level == "beginner").any():
        top = rec.recommend(s, "beginner", 120, top_k=1)[0]
        if top["level"] != "beginner":
            bad.append(names[s])
check("beginners get a beginner resource first (when one exists)", not bad)
if bad: print("   exceptions:", bad)

# 4. time matters: more minutes per day -> shorter estimated days and a higher score for the same resource
a = rec.recommend("S41", "intermediate", 30, top_k=1)[0]
b = rec.recommend("S41", "intermediate", 300, top_k=1)[0]
check("more daily time raises the score of a long resource", b["score"] > a["score"])

# 5. feedback matters: 10 'not helpful' votes push a resource down
top = rec.recommend("S01", "beginner", 120)[0]["resource_id"]
fb_bad = pd.DataFrame({"resource_id": [top] * 10, "helpful": [0] * 10})
rec_fb = ResourceRecommender(resources, feedback=fb_bad)
check("a resource with 10 'not helpful' votes loses its top rank",
      rec_fb.recommend("S01", "beginner", 120)[0]["resource_id"] != top)

# 6. one vote must not decide: a single 'not helpful' vote should not flip the top rank
fb_one = pd.DataFrame({"resource_id": [top], "helpful": [0]})
rec_one = ResourceRecommender(resources, feedback=fb_one)
check("a single vote does not flip the ranking", rec_one.recommend("S01", "beginner", 120)[0]["resource_id"] == top)

PASS - unknown skill returns an empty list
PASS - all scores are between 0 and 1
PASS - top_k=1 returns one resource
PASS - beginners get a beginner resource first (when one exists)
PASS - more daily time raises the score of a long resource
PASS - a resource with 10 'not helpful' votes loses its top rank
PASS - a single vote does not flip the ranking


## 3. Feedback re-ranking demo
The votes below are **simulated** just to show the mechanism. On the website they come from the
"helpful / not helpful" buttons (`resource_feedback` table).

In [5]:
skill = "S20"   # Pandas
before = rec.recommend(skill, "beginner", 120)
worst = before[0]["resource_id"]
sim_fb = pd.DataFrame({"resource_id": [worst] * 8 + [before[1]["resource_id"]] * 8,
                       "helpful":     [0] * 8 + [1] * 8})
after = ResourceRecommender(resources, feedback=sim_fb).recommend(skill, "beginner", 120)
print("BEFORE:", [(r["title"], r["score"]) for r in before])
print("AFTER :", [(r["title"], r["score"]) for r in after])
print(after[0]["reasons"])

BEFORE: [('Kaggle Learn: Pandas', 0.856), ('Pandas: 10 Minutes to Pandas', 0.827)]
AFTER : [('Pandas: 10 Minutes to Pandas', 0.893), ('Kaggle Learn: Pandas', 0.789)]
['matches your level', 'about 0.5 days at 120 min/day', '83% of 8 learners found it helpful']


## 4. How sensitive is the ranking to the weights?
The weights are chosen by hand, so we should check that the top recommendation does not change wildly
if the weights change a little. We compare the top-1 resource across different weight sets.

In [6]:
weight_sets = {
    "default":     DEFAULT_WEIGHTS,
    "level-heavy": {"level": 0.6, "time": 0.1, "type": 0.1, "feedback": 0.15, "free": 0.05},
    "time-heavy":  {"level": 0.25, "time": 0.4, "type": 0.1, "feedback": 0.2, "free": 0.05},
    "type-heavy":  {"level": 0.3, "time": 0.15, "type": 0.35, "feedback": 0.15, "free": 0.05},
}
rows = []
for lv in LEVELS:
    base = {s: ResourceRecommender(resources, weights=weight_sets["default"]).recommend(s, lv, 120, 1)[0]["resource_id"]
            for s in skills.skill_id}
    for name, w in weight_sets.items():
        r = ResourceRecommender(resources, weights=w)
        same = sum(r.recommend(s, lv, 120, 1)[0]["resource_id"] == base[s] for s in skills.skill_id)
        rows.append({"user level": lv, "weights": name, "same top-1 as default (%)": round(100 * same / len(skills))})
pd.DataFrame(rows)

,user level,weights,same top-1 as default (%)
0,beginner,default,100
1,beginner,level-heavy,98
2,beginner,time-heavy,93
3,beginner,type-heavy,98
4,intermediate,default,100
5,intermediate,level-heavy,98
6,intermediate,time-heavy,98
7,intermediate,type-heavy,95
8,advanced,default,100
9,advanced,level-heavy,100


## 5. Human evaluation sheet
Real evaluation: ask classmates "would this resource help this person?" (1-5). The cell below creates the
sheet. Share it, collect ratings, then compute the average rating (cell after).

In [7]:
personas = [("beginner", 60), ("beginner", 180), ("intermediate", 120), ("advanced", 240)]
sample_skills = ["S01", "S06", "S09", "S20", "S24", "S30", "S31", "S37"]
rows = []
for s in sample_skills:
    for lv, mins in personas:
        top = rec.recommend(s, lv, mins, top_k=1)[0]
        rows.append({"skill": names[s], "user_level": lv, "minutes_per_day": mins,
                     "recommended_resource": top["title"], "type": top["type"], "resource_level": top["level"],
                     "rating_1_to_5": ""})
sheet = pd.DataFrame(rows)
sheet.to_csv("../data/resource_rating_sheet.csv", index=False)
print(len(sheet), "rows saved to ../data/resource_rating_sheet.csv")
sheet.head(8)

32 rows saved to ../data/resource_rating_sheet.csv


,skill,user_level,minutes_per_day,recommended_resource,type,resource_level,rating_1_to_5
0,Python,beginner,60,Kaggle Learn: Python,course,beginner,
1,Python,beginner,180,Kaggle Learn: Python,course,beginner,
2,Python,intermediate,120,Python Official Tutorial,docs,beginner,
3,Python,advanced,240,Python Official Tutorial,docs,beginner,
4,JavaScript,beginner,60,JavaScript.info,docs,beginner,
5,JavaScript,beginner,180,JavaScript.info,docs,beginner,
6,JavaScript,intermediate,120,MDN: JavaScript Guide,docs,intermediate,
7,JavaScript,advanced,240,MDN: JavaScript Guide,docs,intermediate,


In [8]:
# Run this after classmates fill the rating_1_to_5 column (the sheet may come back as several files)
rated = pd.read_csv("../data/resource_rating_sheet.csv")
rated["rating_1_to_5"] = pd.to_numeric(rated["rating_1_to_5"], errors="coerce")
if rated["rating_1_to_5"].notna().any():
    print("average rating:", round(rated["rating_1_to_5"].mean(), 2), "from", rated["rating_1_to_5"].notna().sum(), "ratings")
    print(rated.groupby("user_level")["rating_1_to_5"].mean().round(2))
else:
    print("No ratings yet - share the sheet with classmates first.")

No ratings yet - share the sheet with classmates first.
